# Sentiment Analysis Workflow for Employee Email Data

This notebook implements a complete workflow for analyzing employee sentiment from email data, including data cleaning, feature engineering, sentiment scoring, employee ranking, flight risk identification, and predictive modeling. Each section is well-commented and includes observations and explanations to ensure clarity and reproducibility.

## 1. Import Required Libraries

Import all necessary libraries for data processing, visualization, and machine learning. Each import is commented for clarity.

In [ ]:
# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Natural Language Processing
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Utility
import os

# Download NLTK resources if not already present
nltk.download('vader_lexicon')

## 2. Load and Explore the Dataset

In this section, we load the main dataset and perform initial exploration to understand its structure, size, and basic statistics. Observations are provided to guide further processing.

In [ ]:
# Load the labeled email dataset
df = pd.read_csv('data/test_labeled.csv')

# Display the first few rows
display(df.head())

# Show dataset shape and columns
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Show basic statistics for numeric columns
display(df.describe(include='all'))

# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Observation:
# The dataset contains email messages with columns such as subject, body, date, from, and sentiment.
# We will use this information for further analysis and modeling.

## 3. Data Preprocessing

In this section, we clean and preprocess the data, handling missing values and ensuring correct data types. This step is crucial for reliable analysis and modeling.

In [ ]:
# Convert 'date' to datetime and drop rows with missing dates
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])

# Fill missing values in 'body' with empty string
df['body'] = df['body'].fillna('')

# Show updated info and missing values
display(df.info())
print("\nMissing values after preprocessing:")
print(df.isnull().sum())

# Observation:
# The data is now cleaned and ready for feature engineering. All rows have valid dates and message bodies.

## 4. Feature Engineering

We create new features such as message length, word count, and monthly message statistics. These features are used for downstream modeling and analysis.

In [ ]:
# Feature engineering: message length and word count
df['msg_length'] = df['body'].astype(str).apply(len)
df['word_count'] = df['body'].astype(str).apply(lambda x: len(x.split()))
df['month'] = df['date'].dt.to_period('M').astype(str)

# Aggregate features per employee per month
monthly_features = df.groupby(['from', 'month']).agg(
    message_count = ('body', 'count'),
    avg_msg_length = ('msg_length', 'mean'),
    avg_word_count = ('word_count', 'mean'),
).reset_index()

display(monthly_features.head())

# Observation:
# These features will be used for modeling and to understand employee communication patterns.

## 5. Model Selection and Training

We use linear regression to predict monthly sentiment scores based on engineered features. The data is split into training and testing sets to evaluate model performance.

In [ ]:
# Load monthly sentiment scores
df_scores = pd.read_csv('data/sentiment_scores.csv')
df_scores['month'] = df_scores['month'].astype(str)

# Merge features and target
data = pd.merge(monthly_features, df_scores, left_on=['from', 'month'], right_on=['employee', 'month'])

# Prepare features and target
X = data[['message_count', 'avg_msg_length', 'avg_word_count']]
y = data['sentiment_score']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit linear regression
model = LinearRegression()
model.fit(X_train, y_train)

print("Model training complete.")

## 6. Model Evaluation

We evaluate the linear regression model using mean squared error (MSE) and R-squared (R²) metrics. Feature importances are also interpreted.

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R^2 Score: {r2:.2f}")

print("\nFeature importances (coefficients):")
for name, coef in zip(['message_count', 'avg_msg_length', 'avg_word_count'], model.coef_):
    print(f"{name}: {coef:.3f}")

# Observation:
# The model's performance and feature importances help us understand which factors most influence sentiment scores.

## 7. Saving and Loading Models

Persisting trained models allows for future use without retraining. Here, we demonstrate saving and loading the linear regression model using joblib.

In [ ]:
import joblib

# Save the model
joblib.dump(model, 'linear_regression_model.joblib')
print("Model saved as 'linear_regression_model.joblib'.")

# Load the model (example)
loaded_model = joblib.load('linear_regression_model.joblib')
print("Model loaded successfully.")

## 8. Supporting Files Integration

This section documents the supporting Python scripts used in this workflow. Each script automates a key part of the analysis pipeline and can be run independently or called from the notebook as needed.

- **model.py**: Performs sentiment labeling on raw messages using NLTK VADER and saves the labeled data.
- **eda.py**: Provides functions for exploratory data analysis, including visualizations of sentiment distribution and trends.
- **score_calculation.py**: Aggregates sentiment scores per employee per month, used as the main target for modeling and ranking.
- **ranking.py**: Ranks employees by monthly sentiment scores, identifying top positive and negative contributors.
- **flight_risk.py**: Flags employees as flight risks if they send 4+ negative messages in any rolling 30-day window.
- **linear_model.py**: Implements the linear regression model for predicting sentiment scores from engineered features.

Below are code snippets showing how to use these scripts programmatically.

In [ ]:
# Example: Run supporting scripts from the notebook
# Sentiment labeling (model.py)
# %run src/model.py

# Exploratory Data Analysis (eda.py)
# %run src/eda.py

# Monthly sentiment score calculation (score_calculation.py)
# %run src/score_calculation.py

# Employee ranking (ranking.py)
# %run src/ranking.py

# Flight risk identification (flight_risk.py)
# %run src/flight_risk.py

# Linear regression modeling (linear_model.py)
# %run src/linear_model.py

# Uncomment the lines above to execute each script as part of the workflow.